In [0]:
%run ./00_setup

In [0]:
import great_expectations as gx
from great_expectations.core.batch import RuntimeBatchRequest
import datetime as dt

orders_df = (
    spark.table("workspace.bronze.orders")
)

In [0]:
import great_expectations as gx
from great_expectations.core.batch import RuntimeBatchRequest
import datetime as dt

context = gx.get_context()

suite_name = "orders_suite"
context.add_or_update_expectation_suite(expectation_suite_name=suite_name)

datasource_config = {
    "name": "spark_orders_ds",
    "class_name": "Datasource",
    "execution_engine": {
        "class_name": "SparkDFExecutionEngine",
        "persist": False
    },
    "data_connectors": {
        "default_runtime_data_connector_name": {
            "class_name": "RuntimeDataConnector",
            "batch_identifiers": ["default_identifier_name"]
        }
    }
}

datasource = context.add_datasource(**datasource_config)

batch_request = RuntimeBatchRequest(
    datasource_name="spark_orders_ds",
    data_connector_name="default_runtime_data_connector_name",
    data_asset_name="orders_asset",
    runtime_parameters={"batch_data": orders_df},
    batch_identifiers={"default_identifier_name": "orders_batch"}
)

validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=suite_name
)

validator.expect_column_values_to_not_be_null("order_id")
validator.expect_column_values_to_be_unique("order_id")
validator.expect_column_values_to_not_be_null("customer_id")

validator.expect_column_values_to_be_in_set("country", ["ES", "MX", "AR", "CO", "CL"])
validator.expect_column_values_to_be_in_set("currency", ["EUR", "MXN", "ARS", "COP", "CLP"])

validator.expect_column_values_to_be_between("quantity", min_value=1, max_value=100)
validator.expect_column_values_to_be_between("unit_price", min_value=0.01, max_value=10000)
validator.expect_column_values_to_be_between("order_total", min_value=0.01, max_value=1000000)

validator.expect_column_values_to_be_in_set("payment_method", ["CARD", "TRANSFER", "CASH", "PAYPAL"])
validator.expect_column_values_to_be_in_set("order_status", ["CREATED", "PAID", "CANCELLED", "REFUNDED"])

validator.expect_column_values_to_match_regex(
    "email",
    r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
    mostly=0.99
)


validator.expect_column_values_to_be_between(
    "order_ts",
    min_value="2022-01-01 00:00:00",
    max_value=str(dt.datetime.utcnow())
)

validator.save_expectation_suite(discard_failed_expectations=False)
orders_result = validator.validate()

display(orders_result.to_json_dict())

save_json(
    f"{VALIDATION_PATH}/orders_validation.json",
    orders_result.to_json_dict()
)

In [0]:
import pandas as pd
import great_expectations as gx
from great_expectations.core.batch import RuntimeBatchRequest
import datetime as dt

# 1. Cargar los datos con Pandas
# Puedes usar pd.read_csv, pd.read_parquet, etc.


orders_df = (
    spark.table("workspace.bronze.orders")
)

orders_df = orders_df.toPandas()

# IMPORTANTE: Asegúrate de que la columna de fecha sea tipo datetime
orders_df['order_ts'] = pd.to_datetime(orders_df['order_ts'])

# 2. Inicializar el contexto
context = gx.get_context()

# 3. Crear o actualizar la Suite
suite_name = "orders_pandas_suite"
context.add_or_update_expectation_suite(expectation_suite_name=suite_name)

# 4. Configurar el Datasource para PANDAS
datasource_config = {
    "name": "pandas_orders_ds",
    "class_name": "Datasource",
    "execution_engine": {
        "class_name": "PandasExecutionEngine", # <-- Cambio clave aquí
    },
    "data_connectors": {
        "default_runtime_data_connector_name": {
            "class_name": "RuntimeDataConnector",
            "batch_identifiers": ["default_identifier_name"]
        }
    }
}

context.add_datasource(**datasource_config)

# 5. Crear el Batch Request
batch_request = RuntimeBatchRequest(
    datasource_name="pandas_orders_ds",
    data_connector_name="default_runtime_data_connector_name",
    data_asset_name="orders_asset",
    runtime_parameters={"batch_data": orders_df}, # Pasamos el DataFrame de Pandas
    batch_identifiers={"default_identifier_name": "orders_batch_pandas"}
)

validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=suite_name
)

# --- 6. DEFINICIÓN DE REGLAS (Las mismas que en Spark) ---

validator.expect_column_values_to_not_be_null("order_id")
validator.expect_column_values_to_be_unique("order_id")
validator.expect_column_values_to_not_be_null("customer_id")

validator.expect_column_values_to_be_in_set("country", ["ES", "MX", "AR", "CO", "CL"])
validator.expect_column_values_to_be_in_set("currency", ["EUR", "MXN", "ARS", "COP", "CLP"])

validator.expect_column_values_to_be_between("quantity", min_value=1, max_value=100)
validator.expect_column_values_to_be_between("unit_price", min_value=0.01, max_value=10000)
validator.expect_column_values_to_be_between("order_total", min_value=0.01, max_value=1000000)

validator.expect_column_values_to_be_in_set("payment_method", ["CARD", "TRANSFER", "CASH", "PAYPAL"])
validator.expect_column_values_to_be_in_set("order_status", ["CREATED", "PAID", "CANCELLED", "REFUNDED"])

validator.expect_column_values_to_match_regex(
    "email",
    r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
    mostly=0.99
)

# Validación de fecha (Pandas usa objetos datetime)
validator.expect_column_values_to_be_between(
    "order_ts",
    min_value=pd.Timestamp("2022-01-01"),
    max_value=pd.Timestamp(dt.datetime.now())
)

# 7. Guardar y Ejecutar
validator.save_expectation_suite(discard_failed_expectations=False)
orders_result = validator.validate()



In [0]:
# 8. Resultados
print(orders_result.to_json_dict())

# Si tienes la función save_json definida:
# save_json(f"orders_validation_pandas.json", orders_result.to_json_dict())